# Trade Promotion Analytics & Lift Modelling Pipeline
**Author:** Bawelile Gule · Data Scientist  
**Stack:** PySpark · Spark SQL · Delta Lake · XGBoost · MLflow · SHAP · Databricks-compatible  
[![Portfolio](https://img.shields.io/badge/Portfolio-Bawelile.github.io-1A5276?style=flat-square)](https://Bawelile.github.io)

---
## Business Problem

CPG and beverage companies spend 20-25% of revenue on trade promotions — discounts, displays, and feature ads given to retail banners (Kroger, Publix, Walmart) to drive volume. Most promotional spend is planned on historical averages, not predicted lift.

This pipeline predicts **incremental promotional lift** by retailer banner, product, and promotional mechanic — separating true incrementality from pulled-forward demand — using a Medallion architecture on Databricks-compatible infrastructure.

## Architecture
```
Bronze: raw retailer POS feeds (multiple banners)
   ↓
Silver: baseline demand estimation + promo flagging (Spark SQL)
   ↓
Gold: lift model features (XGBoost + SHAP)
   ↓
Predicted lift by banner / product / mechanic + business $ impact
```

---
## 0 · Setup

In [ ]:
%%capture
!pip install pyspark delta-spark xgboost shap mlflow plotly --quiet

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import xgboost as xgb
import shap

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
from pyspark.sql import SparkSession, functions as F, Window
from delta import configure_spark_with_delta_pip

builder = (SparkSession.builder.appName('TradePromoLift')
    .config('spark.sql.extensions','io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog','org.apache.spark.sql.delta.catalog.DeltaCatalog'))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

BANNERS  = ['Kroger','Publix','WalmartEquiv','TargetEquiv']
PRODUCTS = ['Cola_12pk','Cola_2L','DietCola_12pk','Sparkling_8pk','Juice_6pk']
MECHANICS = ['Price_Reduction','Feature_Ad','End_Cap_Display','Multi_Buy','Feature_Plus_Display']

print('Spark + Delta Lake session ready.')
print(f'Banners: {BANNERS}')
print(f'Products: {PRODUCTS}')
print(f'Mechanics: {MECHANICS}')

---
## 1 · Bronze layer — simulate multi-banner retailer POS feeds

In production these would be separate feeds per bottler/retailer relationship. Here we generate realistic weekly POS data across 4 banners × 5 products × 2 years, with embedded promotional events.

In [ ]:
np.random.seed(42)
weeks = pd.date_range('2023-01-02', '2024-12-30', freq='W-MON')
rows = []
for banner in BANNERS:
    banner_scale = {'Kroger':1.4,'Publix':1.0,'WalmartEquiv':1.8,'TargetEquiv':0.9}[banner]
    for product in PRODUCTS:
        prod_base = {'Cola_12pk':800,'Cola_2L':500,'DietCola_12pk':400,'Sparkling_8pk':350,'Juice_6pk':300}[product]
        base = prod_base * banner_scale
        for i, wk in enumerate(weeks):
            seasonal = 1 + 0.15*np.sin(2*np.pi*i/52)
            # Promotions: ~22% of weeks, varying mechanic
            is_promo = np.random.random() < 0.22
            mechanic = np.random.choice(MECHANICS) if is_promo else 'None'
            lift_map = {'Price_Reduction':0.18,'Feature_Ad':0.25,'End_Cap_Display':0.30,
                        'Multi_Buy':0.22,'Feature_Plus_Display':0.42,'None':0.0}
            true_lift = lift_map[mechanic] + np.random.normal(0,0.03)
            baseline_units = base*seasonal*(1+np.random.normal(0,0.06))
            units = baseline_units*(1+max(true_lift,0))
            price = {'Cola_12pk':6.99,'Cola_2L':2.49,'DietCola_12pk':6.99,'Sparkling_8pk':5.49,'Juice_6pk':4.99}[product]
            if is_promo and mechanic=='Price_Reduction': price *= 0.80
            elif is_promo: price *= 0.92
            rows.append(dict(week=wk, banner=banner, product=product,
                              units_sold=max(round(units),0), price=round(price,2),
                              is_promo=int(is_promo), mechanic=mechanic,
                              true_baseline=round(baseline_units,1)))

raw_pdf = pd.DataFrame(rows)
bronze_df = spark.createDataFrame(raw_pdf)
bronze_df.write.format('delta').mode('overwrite').save('/tmp/delta/bronze_promo')
print(f'Bronze layer: {bronze_df.count():,} rows across {len(BANNERS)} banners × {len(PRODUCTS)} products × {len(weeks)} weeks')
bronze_df.show(5)

---
## 2 · Silver layer — baseline demand estimation (Spark SQL)

Estimate what demand *would have been* without a promotion, using non-promo weeks' rolling average per banner/product. This baseline is the foundation for measuring true incremental lift.

In [ ]:
bronze_df.createOrReplaceTempView('bronze_promo')

silver_df = spark.sql('''
    SELECT *,
        AVG(CASE WHEN is_promo=0 THEN units_sold END) OVER (
            PARTITION BY banner, product
            ORDER BY week
            ROWS BETWEEN 8 PRECEDING AND 1 PRECEDING
        ) AS rolling_baseline_8wk,
        AVG(units_sold) OVER (
            PARTITION BY banner, product
            ORDER BY week
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS recent_avg_3wk,
        LAG(units_sold, 52) OVER (
            PARTITION BY banner, product ORDER BY week
        ) AS units_yoy
    FROM bronze_promo
''')

# Compute observed lift vs estimated baseline
silver_df = silver_df.withColumn(
    'estimated_baseline',
    F.coalesce(F.col('rolling_baseline_8wk'), F.col('recent_avg_3wk'))
).withColumn(
    'observed_lift_pct',
    F.when(F.col('estimated_baseline') > 0,
           (F.col('units_sold')-F.col('estimated_baseline'))/F.col('estimated_baseline')
    ).otherwise(0)
)

silver_df.write.format('delta').mode('overwrite').save('/tmp/delta/silver_promo')
print('Silver layer written — baseline + observed lift computed via Spark SQL window functions')
silver_df.filter('is_promo=1').select('banner','product','week','mechanic','units_sold',
    'estimated_baseline','observed_lift_pct').show(5)

---
## 3 · Gold layer — feature engineering for lift model

In [ ]:
gold_pdf = silver_df.toPandas().dropna(subset=['estimated_baseline'])

# One-hot encode categorical features
gold_pdf['week_num'] = pd.to_datetime(gold_pdf['week']).dt.isocalendar().week.astype(int)
gold_pdf['month'] = pd.to_datetime(gold_pdf['week']).dt.month

cat_cols = ['banner','product','mechanic']
gold_encoded = pd.get_dummies(gold_pdf, columns=cat_cols, drop_first=False)

feature_cols = [c for c in gold_encoded.columns if c.startswith(tuple(cat_cols))] + \
                ['price','week_num','month','estimated_baseline']

gold_encoded.to_parquet('/tmp/gold_promo.parquet')
print(f'Gold layer: {len(gold_encoded):,} rows, {len(feature_cols)} features')
print(f'Promo weeks: {gold_pdf["is_promo"].sum():,} | Non-promo weeks: {(gold_pdf["is_promo"]==0).sum():,}')

---
## 4 · Lift model — XGBoost

Predict observed_lift_pct for promo weeks using mechanic, banner, product, price, and seasonality. This is the model a bottler would use to forecast expected lift *before* committing promotional spend.

In [ ]:
import mlflow
mlflow.set_experiment('trade_promo_lift')

promo_data = gold_encoded[gold_encoded['is_promo']==1].copy()
X = promo_data[feature_cols]
y = promo_data['observed_lift_pct']

split = int(len(X)*0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

with mlflow.start_run(run_name='lift_xgb_v1'):
    model = xgb.XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                              subsample=0.85, colsample_bytree=0.8, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = float(np.mean(np.abs(preds-y_test)))
    mlflow.log_params({'n_estimators':300,'max_depth':4,'learning_rate':0.05})
    mlflow.log_metric('mae_lift_pct', mae)

print(f'Lift prediction MAE: {mae:.3f} ({mae*100:.1f} percentage points)')
print(f'Avg actual lift in test set: {y_test.mean()*100:.1f}%')
print(f'Avg predicted lift: {preds.mean()*100:.1f}%')

---
## 5 · SHAP explainability — what drives lift?

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
mean_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_cols).sort_values(ascending=False)

print('Top 8 drivers of promotional lift:')
for feat, val in mean_shap.head(8).items():
    print(f'  {feat:<28}: {val:.4f}')

---
## 6 · Lift by mechanic and banner — business view

In [ ]:
mechanic_lift = gold_pdf[gold_pdf['is_promo']==1].groupby('mechanic')['observed_lift_pct'].mean().sort_values()
banner_lift = gold_pdf[gold_pdf['is_promo']==1].groupby('banner')['observed_lift_pct'].mean().sort_values()

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Average lift by promotional mechanic', 'Average lift by retail banner'))

fig.add_trace(go.Bar(x=mechanic_lift.values*100, y=mechanic_lift.index,
    orientation='h', marker_color='#1A5276', showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=banner_lift.values*100, y=banner_lift.index,
    orientation='h', marker_color='#2E86C1', showlegend=False), row=1, col=2)

fig.update_layout(height=380, title_text='Trade promotion lift analysis',
    plot_bgcolor='white')
fig.update_xaxes(title_text='Avg lift %', showgrid=False)
fig.show()

print('Best mechanic:', mechanic_lift.idxmax(), f'({mechanic_lift.max()*100:.1f}% avg lift)')
print('Best banner:', banner_lift.idxmax(), f'({banner_lift.max()*100:.1f}% avg lift)')

---
## 7 · Business impact — promo ROI estimator

In [ ]:
def promo_roi(banner, product, mechanic, price, baseline_units, promo_cost_per_unit):
    """Estimate incremental revenue and ROI for a planned promotion."""
    row = {c:0 for c in feature_cols}
    for c in feature_cols:
        if c == f'banner_{banner}': row[c]=1
        if c == f'product_{product}': row[c]=1
        if c == f'mechanic_{mechanic}': row[c]=1
    row['price']=price; row['week_num']=26; row['month']=6
    row['estimated_baseline']=baseline_units
    X_new = pd.DataFrame([row])[feature_cols]
    pred_lift = model.predict(X_new)[0]
    incremental_units = baseline_units * pred_lift
    incremental_revenue = incremental_units * price
    promo_spend = (baseline_units+incremental_units) * promo_cost_per_unit
    roi = (incremental_revenue - promo_spend) / promo_spend if promo_spend>0 else 0
    return dict(predicted_lift_pct=round(pred_lift*100,1),
                 incremental_units=round(incremental_units),
                 incremental_revenue=round(incremental_revenue,2),
                 promo_spend=round(promo_spend,2),
                 roi_pct=round(roi*100,1))

scenario = promo_roi('Kroger','Cola_12pk','Feature_Plus_Display',6.99,800,0.75)
print('Scenario: Kroger | Cola 12pk | Feature + Display | $6.99 | promo cost $0.75/unit')
for k,v in scenario.items():
    print(f'  {k}: {v}')

---
## 8 · Production notes — Databricks deployment

In [ ]:
print('''
PRODUCTION DEPLOYMENT — DATABRICKS
====================================
Bronze/Silver/Gold tables persist as managed Delta tables — on Databricks,
swap /tmp/delta/* paths for a catalog.schema.table reference and these
become queryable via Databricks SQL for bottler-facing dashboards.

Each new retailer banner = new partition, not new pipeline.
MLflow tracking integrates natively with Databricks Model Registry —
promo lift model can be versioned and served via a Databricks endpoint.

Next step: connect promo_roi() to a planning UI so bottler commercial
teams can test \"what-if\" promo scenarios before committing spend —
directly mirroring CONA's recommendations + product availability platform.
''')